In [1]:
print("Ok, let's start with RAG demo")

Ok, let's start with RAG demo


In [2]:
# Get file
from pathlib import Path
file_path = Path("../data/information.txt").resolve()
print(f"File path: {file_path}")

File path: D:\Samples\RagDemo\data\information.txt


In [3]:
# Load document
from langchain_community.document_loaders import TextLoader
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")

Loaded 1 document(s)


In [4]:
documents[0]

Document(metadata={'source': 'D:\\Samples\\RagDemo\\data\\information.txt'}, page_content="The Great Gatsby by F. Scott Fitzgerald\nA classic novel set in the Roaring Twenties that explores wealth, love, and the American Dream through the mysterious Jay Gatsby.\n\nTo Kill a Mockingbird by Harper Lee\nA powerful story about racial injustice and moral courage in the American South, narrated through the eyes of a young girl named Scout Finch.\n\n1984 by George Orwell\nA dystopian novel depicting a totalitarian society ruled by surveillance, censorship, and manipulation of truth.\n\nPride and Prejudice by Jane Austen\nA timeless romantic novel that follows Elizabeth Bennet as she navigates love, class, and societal expectations.\n\nThe Hobbit by J.R.R. Tolkien\nAn adventurous fantasy tale about Bilbo Baggins, a hobbit who embarks on an unexpected journey with dwarves and a wizard.\n\nMoby-Dick by Herman Melville\nThe story of Captain Ahab’s obsessive quest to hunt the legendary white whale

In [5]:
# Create chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = text_splitter.split_documents(documents=documents)
print(f"Created {len(chunks)} chunk(s)")

Created 2 chunk(s)


In [6]:
# create embeddings
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv() # load environment variables from .env file
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)


In [7]:
# Just check embedding, not for rag flow
embeddings = embedding_model.embed_documents([chunk.page_content for chunk in chunks])
embeddings

[[-0.0087432861328125,
  0.039306640625,
  -0.00870513916015625,
  0.03790283203125,
  -0.00778961181640625,
  -0.02349853515625,
  -0.042724609375,
  0.02911376953125,
  0.00983428955078125,
  0.0087127685546875,
  0.037933349609375,
  -0.01898193359375,
  -0.09033203125,
  -0.0053253173828125,
  0.08636474609375,
  -0.0033512115478515625,
  -0.0440673828125,
  0.01233673095703125,
  0.0306549072265625,
  0.058502197265625,
  0.027069091796875,
  0.0138092041015625,
  -0.01141357421875,
  -0.03082275390625,
  -0.05804443359375,
  0.05865478515625,
  -0.07733154296875,
  0.06231689453125,
  0.0283355712890625,
  0.01392364501953125,
  -0.02471923828125,
  0.0013942718505859375,
  -0.01214599609375,
  -0.049346923828125,
  0.052001953125,
  -0.0465087890625,
  0.035919189453125,
  -0.032257080078125,
  0.03955078125,
  -0.007843017578125,
  -0.0267333984375,
  -0.07159423828125,
  0.06903076171875,
  -0.01314544677734375,
  0.02978515625,
  0.00650787353515625,
  0.061981201171875,
  -0

In [8]:
# Create vectorstore 
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(documents=chunks, embedding=embedding_model)

In [9]:
# Get relevant documents
relevant_docs = vector_store.similarity_search("story of Captain Ahabâ€™s")

In [10]:
# format relevant documents for better display
def format_relevant_docs(docs):
    return [doc for doc in docs]

In [11]:
# prompt template where context is the relevant documents and question is the user query
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that answers questions based on the following context: {context}"), 
        ("user", "Question: {question}")
    ])

In [12]:
# Instantiate the model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [13]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [14]:
# Create RAG chain
rag_chain = prompt | llm | output_parser

In [15]:
# Invoke the RAG chain
context = format_relevant_docs(relevant_docs)
question = "What is the story of Captain Ahabâ€™s?" # Happy test case
# question = "What is the story of shubham?" # Negative test case

rag_chain.invoke(
    {"context": context, "question": question}
)

'The story of Captain Ahab is depicted in the novel "Moby-Dick" by Herman Melville. Captain Ahab is the main character in the novel, and his story revolves around his obsessive quest to hunt the legendary white whale, Moby Dick. Ahab\'s relentless pursuit of the whale drives the narrative of the novel, exploring themes of obsession, revenge, and the destructive nature of unchecked ambition.'